# 02 — EDA & Featurization
**Project:** Phase-Field Informed ML for Microstructure Prediction in Ti-Nb-O Refractory Alloys  
**Author:** Anosike Kelechi Kenneth  
**Purpose:** Exploratory data analysis of the bandgap dataset, apply Magpie featurization, and save the feature matrix for modeling.  
**Run order:** Run AFTER 01_data_acquisition.ipynb

---

In [ ]:
# Section 1: Imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from matminer.datasets import load_dataset
from matminer.featurizers.composition import ElementProperty
from matminer.featurizers.conversions import StrToComposition

print('All imports OK')

In [ ]:
# Section 2: Reload dataset (same seed as notebook 01)
# Using relative path - run from inside the repo root or notebooks/ folder
RANDOM_STATE = 42
SAMPLE_SIZE = 20000

print('Reloading matbench-mp-gap subset...')
df_full = load_dataset('matbench_mp_gap')
df = df_full.sample(SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
df['composition_str'] = df['structure'].apply(lambda s: str(s.composition))
print(f'Dataset shape: {df.shape}')

In [ ]:
# Section 3: EDA - target distribution analysis
from scipy import stats

target = df['gap pbe']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Raw histogram
axes[0].hist(target, bins=60, color='steelblue', alpha=0.85, edgecolor='white', density=True)
axes[0].set_xlabel('Band Gap (eV)', fontsize=12)
axes[0].set_title('Raw Distribution', fontsize=12)

# Log transformed
axes[1].hist(np.log1p(target), bins=60, color='teal', alpha=0.85, edgecolor='white', density=True)
axes[1].set_xlabel('log(Band Gap + 1)', fontsize=12)
axes[1].set_title('Log-Transformed', fontsize=12)

# Q-Q plot
stats.probplot(target, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot (Normal)', fontsize=12)

plt.suptitle('Band Gap Distribution Analysis', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/02_distribution_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

stat, p = stats.shapiro(target.sample(500, random_state=42))
print(f'Shapiro-Wilk p-value: {p:.4f} ({"non-normal" if p < 0.05 else "normal"} distribution)')
print(f'Skewness: {target.skew():.3f}')
print(f'Kurtosis: {target.kurtosis():.3f}')

In [ ]:
# Section 4: EDA - class balance (metals vs non-metals)
n_metals = (df['gap pbe'] == 0).sum()
n_nonmetals = (df['gap pbe'] > 0).sum()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Metals (gap=0)', 'Semiconductors/Insulators'], [n_metals, n_nonmetals],
       color=['#1C2B4A', '#0D9488'], alpha=0.85)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Class Balance: Metals vs Non-Metals', fontsize=12)
for i, v in enumerate([n_metals, n_nonmetals]):
    ax.text(i, v + 100, f'{v} ({v/len(df)*100:.1f}%)', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('../figures/02_class_balance.png', dpi=150)
plt.show()

In [ ]:
# Section 5: Magpie featurization
# Convert composition strings to pymatgen Composition objects
print('Converting to pymatgen Composition objects...')
stc = StrToComposition(target_col_id='composition_pmg')
df = stc.featurize_dataframe(df, 'composition_str')

# Apply Magpie features
print('Applying Magpie featurization (this takes ~5-10 minutes)...')
ep = ElementProperty.from_preset('magpie')
t0 = time.time()
df = ep.featurize_dataframe(df, col_id='composition_pmg')
print(f'Featurization complete in {time.time()-t0:.1f}s')
print(f'Number of Magpie features: {len(ep.feature_labels())}')
print(f'Full dataframe shape: {df.shape}')

In [ ]:
# Section 6: Feature correlation heatmap (top 20 features)
feature_cols = ep.feature_labels()
X = df[feature_cols]
y = df['gap pbe']

# Correlation with target - find top 20 most correlated features
corr_with_target = X.corrwith(y).abs().sort_values(ascending=False)
top20_feats = corr_with_target.head(20).index.tolist()

fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = df[top20_feats].corr()
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, ax=ax,
            xticklabels=True, yticklabels=True, linewidths=0.3)
ax.set_title('Correlation Heatmap — Top 20 Magpie Features', fontsize=12)
plt.tight_layout()
plt.savefig('../figures/02_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 features most correlated with band gap:')
print(corr_with_target.head(10).round(4))

In [ ]:
# Section 7: Save feature matrix for modeling notebook
# Save features + target as CSV for notebook 03
save_cols = feature_cols + ['gap pbe']
df[save_cols].to_csv('../data/magpie_features_20k.csv', index=False)
print(f'Saved feature matrix: data/magpie_features_20k.csv')
print(f'Shape: {df[save_cols].shape}')
print(f'\nFeature matrix summary:')
print(f'  Features: {len(feature_cols)}')
print(f'  Samples: {len(df)}')
print(f'  Missing values: {df[feature_cols].isnull().sum().sum()}')
print('\nNext step: Run 03_modeling.ipynb')